# 05 — Posterior Predictive Checks: Evaluación de Sobredispersión

**Bayesian Workflow — Etapa: Model Checking (Gelman et al. 2020)**

Los PPCs evalúan si el modelo es capaz de generar datos similares a los observados.
Para datos de conteo, el foco es en **sobredispersión**: el fenómeno donde la varianza
supera a la media (Var[Y] > E[Y]), que Poisson no puede capturar pero la Binomial Negativa sí.

**Pregunta central:** ¿El modelo Poisson subestima la varianza de los datos de cangrejos?
¿La Binomial Negativa resuelve este problema?

## Prerequisito
Ejecutar primero `03_inferencia_bayesiana.ipynb` para generar `outputs/pois_idata.nc`
y `outputs/neg_idata.nc` con los grupos `posterior_predictive` y `log_likelihood`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az

sns.set_style('whitegrid')
np.random.seed(42)

In [ ]:
# Cargar InferenceData con posterior predictive
idata_pois = az.from_netcdf('../outputs/pois_idata.nc')
idata_nb   = az.from_netcdf('../outputs/neg_idata.nc')

# Datos observados
df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')
y_obs = df['Sa'].values

print(f"Grupos disponibles Poisson:  {list(idata_pois.groups())}")
print(f"Grupos disponibles NegBin:   {list(idata_nb.groups())}")

## 1. Verificación de sobredispersión en los datos observados

Bajo Poisson: E[Y] = Var[Y] → Índice de dispersión D = Var/Mean ≈ 1  
Si D > 1 → sobredispersión (la Binomial Negativa es más apropiada)

In [ ]:
mean_obs = np.mean(y_obs)
var_obs  = np.var(y_obs)
D_obs    = var_obs / mean_obs  # Índice de dispersión (VMR)

print(f"Media observada:              {mean_obs:.3f}")
print(f"Varianza observada:           {var_obs:.3f}")
print(f"Índice de dispersión (D=V/M): {D_obs:.3f}  {'← sobredispersión' if D_obs > 1 else '← equidispersión'}")
print(f"Proporción de ceros:          {np.mean(y_obs == 0):.3f}")
print(f"Valor máximo:                 {np.max(y_obs)}")

## 2. PPC visual — Distribución marginal

`az.plot_ppc` superpone la distribución de los datos replicados y_rep sobre y_obs.
Si el modelo es adecuado, la nube de y_rep debe envolver a y_obs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

az.plot_ppc(idata_pois, ax=axes[0], num_pp_samples=100)
axes[0].set_title('Poisson')
axes[0].set_xlabel('Número de satélites')

az.plot_ppc(idata_nb, ax=axes[1], num_pp_samples=100)
axes[1].set_title('Binomial Negativa')
axes[1].set_xlabel('Número de satélites')

plt.suptitle('PPC: Distribución marginal de y_rep vs y_obs', fontsize=13)
plt.tight_layout()
plt.show()

## 3. PPCs con estadísticos de prueba (test statistics)

Para cada estadístico T, comparamos T(y_obs) con la distribución de T(y_rep).
Si T(y_obs) cae en la cola de T(y_rep) → el modelo no captura esa propiedad.

**Estadísticos para sobredispersión:**
- `std`: ¿captura la dispersión total?
- `var/mean`: índice de dispersión (D=1 bajo Poisson)
- `max`: ¿captura las colas pesadas?
- `mean(y==0)`: ¿captura la inflación de ceros?

In [ ]:
def dispersion_index(y):
    """Índice de dispersión: Var/Mean. Bajo Poisson debe ser ~1."""
    return np.var(y) / np.mean(y)

def prop_zeros(y):
    """Proporción de ceros."""
    return np.mean(y == 0)

test_stats = {
    'std':               np.std,
    'Índice dispersión': dispersion_index,
    'max':               np.max,
    'prop. ceros':       prop_zeros,
}

# TODO: implementar comparación T(y_obs) vs distribución de T(y_rep)
# Para cada modelo (Poisson, NegBin) y cada estadístico
# Calcular Bayesian p-value: P(T(y_rep) >= T(y_obs))

In [ ]:
# Extraer muestras y_rep
y_rep_pois = idata_pois.posterior_predictive['y_rep'].values  # shape: (chains, draws, N)
y_rep_nb   = idata_nb.posterior_predictive['y_rep'].values

# Aplanar chains y draws
y_rep_pois_flat = y_rep_pois.reshape(-1, y_rep_pois.shape[-1])  # (S, N)
y_rep_nb_flat   = y_rep_nb.reshape(-1, y_rep_nb.shape[-1])

print(f"y_rep_pois shape: {y_rep_pois_flat.shape}")
print(f"y_rep_nb shape:   {y_rep_nb_flat.shape}")

In [ ]:
fig, axes = plt.subplots(len(test_stats), 2, figsize=(12, 4 * len(test_stats)))

for row, (stat_name, stat_fn) in enumerate(test_stats.items()):
    t_obs = stat_fn(y_obs)
    
    for col, (y_rep_flat, model_name) in enumerate([
        (y_rep_pois_flat, 'Poisson'),
        (y_rep_nb_flat,   'Binomial Negativa')
    ]):
        t_rep = np.array([stat_fn(y_rep_flat[s]) for s in range(y_rep_flat.shape[0])])
        p_val = np.mean(t_rep >= t_obs)
        
        ax = axes[row, col]
        ax.hist(t_rep, bins=40, color='steelblue', alpha=0.7, label='T(y_rep)')
        ax.axvline(t_obs, color='red', lw=2, label=f'T(y_obs) = {t_obs:.2f}')
        ax.set_title(f'{model_name} — {stat_name}\nBayesian p-value = {p_val:.3f}')
        ax.legend(fontsize=8)

plt.suptitle('PPCs con estadísticos de prueba', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. LOO-PIT — Calibración predictiva

El LOO-PIT evalúa calibración: si el modelo está bien calibrado, los quantiles de
y_obs dentro de la distribución predictiva leave-one-out deben ser uniformes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

az.plot_loo_pit(idata_pois, y='y', ax=axes[0])
axes[0].set_title('Poisson — LOO-PIT')

az.plot_loo_pit(idata_nb, y='y', ax=axes[1])
axes[1].set_title('Binomial Negativa — LOO-PIT')

plt.tight_layout()
plt.show()

## 5. Comparación de modelos (LOO-CV)

`az.compare` usa Leave-One-Out Cross-Validation para comparar la capacidad predictiva
de los modelos. Un ELPD más alto = mejor predicción out-of-sample.

In [ ]:
comparison = az.compare({'Poisson': idata_pois, 'NegBin': idata_nb})
display(comparison)

az.plot_compare(comparison, figsize=(8, 3))
plt.title('Comparación LOO-CV: Poisson vs Binomial Negativa')
plt.show()

## 6. Resumen de evidencia

TODO: tabla comparativa con todos los estadísticos y Bayesian p-values por modelo.

In [ ]:
# Tabla resumen
rows = []
for stat_name, stat_fn in test_stats.items():
    t_obs = stat_fn(y_obs)
    for y_rep_flat, model_name in [
        (y_rep_pois_flat, 'Poisson'),
        (y_rep_nb_flat,   'NegBin')
    ]:
        t_rep = np.array([stat_fn(y_rep_flat[s]) for s in range(y_rep_flat.shape[0])])
        rows.append({
            'Modelo': model_name,
            'Estadístico': stat_name,
            'T(y_obs)': round(t_obs, 3),
            'E[T(y_rep)]': round(np.mean(t_rep), 3),
            'Bayesian p-value': round(np.mean(t_rep >= t_obs), 3),
        })

summary_df = pd.DataFrame(rows).sort_values(['Estadístico', 'Modelo'])
display(summary_df)